# Preparação de Dados

## 1 Byte pair encoding de palavras fora do voculário

Durante a aula vimos que um tokenizador baseado em Byte pair encoding (BPE) é capaz de lidar com palavras fora do vocabulário ao dividir uma palavra em "sub-palavras" que estejam presentes no vocabulário. Na pior das hipóteses a palavra pode ser quebrada em letras individuais.

O texto abaixo é um trecho tirado do primeiro parágrafo do livro "The Time Machine" (H. G. Wells, 1895). Use o Tiktoken (com encoding do gpt2) visto durante a aula para tokenizá-lo e verifique quais palavras não estão presentes no vocabulário e necessitaram ser quebradas em "sub-palavras". Mostre como ficou a divisão de cada uma das palavras originalmente fora do vocabulário após a tokenização.

Por exemplo, a palavra "luxurious":<br>
`luxurious -> ['lux', 'urious']`

In [2]:
time_machine_text = 'The Time Traveller was expounding a recondite matter to us. \
His grey eyes shone and twinkled, and his usually pale face was flushed and animated.'

In [16]:
import tiktoken

tokenizador_tiktoken = tiktoken.get_encoding("gpt2")
print("Texto original do exercicio:")
print(time_machine_text)

texto_time_machine_tokenizado_ids = tokenizador_tiktoken.encode(time_machine_text)
print("\nTexto após passar pelo tokenizador e ficar na forma de tokens ids:")
print(texto_time_machine_tokenizado_ids)
print(type(texto_time_machine_tokenizado_ids))
# posso tentar percorrer um loop for para mostrar palavra por palavra e achar as repartidas:
for token_id in texto_time_machine_tokenizado_ids:
    print(f"Token id: {token_id} -> {tokenizador_tiktoken.decode([token_id])}")


Texto original do exercicio:
The Time Traveller was expounding a recondite matter to us. His grey eyes shone and twinkled, and his usually pale face was flushed and animated.

Texto após passar pelo tokenizador e ficar na forma de tokens ids:
[464, 3862, 43662, 6051, 373, 1033, 9969, 257, 664, 623, 578, 2300, 284, 514, 13, 2399, 13791, 2951, 44193, 290, 665, 676, 992, 11, 290, 465, 3221, 14005, 1986, 373, 44869, 290, 15108, 13]
<class 'list'>
Token id: 464 -> The
Token id: 3862 ->  Time
Token id: 43662 ->  Trave
Token id: 6051 -> ller
Token id: 373 ->  was
Token id: 1033 ->  exp
Token id: 9969 -> ounding
Token id: 257 ->  a
Token id: 664 ->  rec
Token id: 623 -> ond
Token id: 578 -> ite
Token id: 2300 ->  matter
Token id: 284 ->  to
Token id: 514 ->  us
Token id: 13 -> .
Token id: 2399 ->  His
Token id: 13791 ->  grey
Token id: 2951 ->  eyes
Token id: 44193 ->  shone
Token id: 290 ->  and
Token id: 665 ->  tw
Token id: 676 -> ink
Token id: 992 -> led
Token id: 11 -> ,
Token id: 290 -> 

Ficou ruim de ver. Vou tentar voltar para lista desconvertendo em partes e não direto para não reconstruir o texto.

In [18]:
texto_time_machine_tokenizado_subwords = []
for token_id in texto_time_machine_tokenizado_ids:
    texto_time_machine_tokenizado_subwords.append(tokenizador_tiktoken.decode([token_id]))
print(texto_time_machine_tokenizado_subwords)

['The', ' Time', ' Trave', 'ller', ' was', ' exp', 'ounding', ' a', ' rec', 'ond', 'ite', ' matter', ' to', ' us', '.', ' His', ' grey', ' eyes', ' shone', ' and', ' tw', 'ink', 'led', ',', ' and', ' his', ' usually', ' pale', ' face', ' was', ' flushed', ' and', ' animated', '.']


In [24]:
import re
Tokenizando_nivel_palavra_time_machine_text = re.split(r'([,.:;?_!"()\']|--|\s)', time_machine_text)
Tokenizando_nivel_palavra_time_machine_text = [item.strip() for item in Tokenizando_nivel_palavra_time_machine_text if item.strip()]
print(Tokenizando_nivel_palavra_time_machine_text)

['The', 'Time', 'Traveller', 'was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', '.', 'His', 'grey', 'eyes', 'shone', 'and', 'twinkled', ',', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', '.']


Consegui gerar com base no que estudei sobre tokenização 2 vetores:
- Vetor com as "traduções" dos token id para word/subwords de acordo com o vocabulário do gpt-2
- Vetor com as palavras devidamente separadas no texto fornecido de time_machine

Agora, executando um força bruta consigo comparar em ~O(n^2) quais palavras estão presentes no vocabulário do gpt e quais não estão (guardarei estas num vetor).

In [31]:
texto_time_machine_tokenizado_subwords = [item.strip() for item in texto_time_machine_tokenizado_subwords if item.strip()] # precisei tirar uns espaços do tokenizado pelo gpt2 ele mete uns espaços em algumas palavras ;-;
palavras_divididas=[]
for word in Tokenizando_nivel_palavra_time_machine_text:
    flag = 0
    for subword in texto_time_machine_tokenizado_subwords:
        if(word == subword):
            flag = 1
            break
    if(flag==0):
        palavras_divididas.append(word)
print("Palavras que foram divididas em subwords:")
print(palavras_divididas)

Palavras que foram divididas em subwords:
['Traveller', 'expounding', 'recondite', 'twinkled']


Para finalizar (to treinando legal professor desculpa a redundância)

In [41]:
for word in palavras_divididas:
    lista_subwords_traduzidas = []
    lista_subwords_tokenizada_id = tokenizador_tiktoken.encode(f" {word}") # Precisei adicionar um espaço na frente do "Traveller" para ele dividir igual o gpt2 ficando ['Trave', 'ller']. Sem o espaço na frente ele dividia ['T', 'rave', 'ller']. gpt2 lida com palavras com espaço na frente diferente das sem espaço. LEGAL!!
    for subword in lista_subwords_tokenizada_id:
        lista_subwords_traduzidas.append(tokenizador_tiktoken.decode([subword]).strip()) # depois removo o espaço da frente para deixar bonitin
    print(f"A palavra {word} -> {lista_subwords_traduzidas}")

A palavra Traveller -> ['Trave', 'ller']
A palavra expounding -> ['exp', 'ounding']
A palavra recondite -> ['rec', 'ond', 'ite']
A palavra twinkled -> ['tw', 'ink', 'led']


## 2 Data loader com diferentes tamanhos de contexto e strides

Durante a aula, vimos como criar um data loader pra treinar uma LLM através da tarefa de prever o próximo token. No caso, o input `x` é uma sequência de tokens e o alvo `y` é o próximo token da sequência `x`. O data loader cria uma janela deslizante que percorre todo o texto, gerando inúmeros exemplos de treino `x, y`. A quantidade de dados de treino gerada pelo data loader vai variar de acordo com o tamanho de `x` (`max_length`) e o tanto que a janela irá deslizar (`stride`) ao longo do texto.

Use o data loader visto durante a aula para tokenizar o texto abaixo com duas configurações distintas:
- `batch_size=4, max_length=2, stride=1`
- `batch_size=4, max_length=6, stride=2`

E responda, quantos exemplos de treino cada configuração o data loader gerou? Lembre-se que cada batch pode conter até 4 exemplos de treino.

In [3]:
time_machine_text = "The Time Traveller (for so it will be convenient to speak of him) \
was expounding a recondite matter to us. His grey eyes shone and \
twinkled, and his usually pale face was flushed and animated. The \
fire burned brightly, and the soft radiance of the incandescent \
lights in the lilies of silver caught the bubbles that flashed and \
passed in our glasses. Our chairs, being his patents, embraced and \
caressed us rather than submitted to be sat upon, and there was that \
luxurious after-dinner atmosphere when thought roams gracefully \
free of the trammels of precision. And he put it to us in this \
way--marking the points with a lean forefinger--as we sat and lazily \
admired his earnestness over this new paradox (as we thought it) \
and his fecundity."

In [4]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# SEU CÓDIGO COM A CLASSE DO DATASET E A FUNÇÃO DO DATA LOADER

In [ ]:
# CHAME O DATA LOADER COM TEXTO ACIMA COM A 1ª CONFIGURAÇÃO
# E CONTE OS EXEMPLOS DE TREINO


In [ ]:
# CHAME O DATA LOADER COM TEXTO ACIMA COM A 2ª CONFIGURAÇÃO
# E CONTE OS EXEMPLOS DE TREINO
